# 🧭 Busca em Profundidade (DFS)

A **Busca em Profundidade** (Depth-First Search — DFS) explora um grafo avançando sempre que possível para um vértice ainda não visitado, recuando (backtracking) quando necessário. O algoritmo constrói uma **floresta DFS** (uma árvore por componente) e registra tempos de descoberta e término de cada vértice.

## 🎯 Estratégia e Funcionamento
- Avança o mais fundo possível a partir do vértice atual.
- Quando um vértice esgota seus vizinhos, a busca recua para o seu antecessor.
- Repete até explorar todos os vértices alcançáveis; se restarem brancos, reinicia em outro vértice, formando uma floresta.

## 🎨 Cores e Estados
- Branco: não descoberto
- Cinza: descoberto, em processamento (há vizinhos a explorar)
- Preto: finalizado (todos os vizinhos examinados)

## ⏱️ Tempos de Descoberta e Término
Para cada vértice v:
- d[v]: instante em que v é descoberto (torna-se cinza).
- t[v]: instante em que v é finalizado (torna-se preto).
São inteiros entre 1 e 2|V| (cada vértice gera dois eventos).

Propriedade de parênteses: ao modelar d[v] como '(', t[v] como ')', os intervalos ficam apropriadamente aninhados.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from typing import Any, Dict, List, Tuple

def dfs(G: nx.Graph) -> Tuple[Dict[Any,str], Dict[Any,int], Dict[Any,int], Dict[Any,Any], List[Tuple[Any,Any]], List[Any]]:
    """
    Executa DFS (recursiva) sobre todos os vértices de G (floresta).
    Retorna (cor, d, t, pai, arestas_floresta, ordem_descoberta).
    - arestas_floresta: lista de arestas (u,v) quando v é descoberto a partir de u.
    Suporta nx.Graph e nx.DiGraph.
    """
    cor: Dict[Any,str] = {v: 'white' for v in G.nodes()}
    d: Dict[Any,int] = {v: 0 for v in G.nodes()}
    t: Dict[Any,int] = {v: 0 for v in G.nodes()}
    pai: Dict[Any,Any] = {v: None for v in G.nodes()}
    tempo = 0
    arestas_floresta: List[Tuple[Any,Any]] = []
    ordem_descoberta: List[Any] = []

    def dfs_visit(u):
        nonlocal tempo
        cor[u] = 'gray'
        tempo += 1
        d[u] = tempo
        ordem_descoberta.append(u)
        for v in G.neighbors(u):
            if cor[v] == 'white':
                pai[v] = u
                arestas_floresta.append((u, v))
                dfs_visit(v)
        cor[u] = 'black'
        tempo += 1
        t[u] = tempo

    for u in G.nodes():
        if cor[u] == 'white':
            dfs_visit(u)

    return cor, d, t, pai, arestas_floresta, ordem_descoberta

def floresta_dfs_subgrafo(G: nx.Graph, arestas_floresta):
    T = nx.DiGraph()
    T.add_nodes_from(G.nodes())
    T.add_edges_from(arestas_floresta)
    return T

## 🧪 Exemplo 1: Grafo não dirigido

In [ ]:
G = nx.Graph()
G.add_edges_from([(1,2),(1,3),(2,4),(2,5),(3,6),(5,6),(6,7)])
cor, d, t, pai, arestas, ordem = dfs(G)
print('Ordem de descoberta:', ordem)
print('Tempos de descoberta d:', d)
print('Tempos de término t:', t)
print('Pais:', pai)

T = floresta_dfs_subgrafo(G, arestas)
pos = nx.spring_layout(G, seed=7)
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True, constrained_layout=True)
plt.subplot(1,2,1)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=1000,
        font_size=12, font_weight='bold', ax=ax)
ax.set_title('Grafo original')

plt.subplot(1,2,2)
nx.draw(G, pos, with_labels=True, node_color='lightgreen', node_size=1000,
        font_size=12, font_weight='bold', ax=ax)
nx.draw_networkx_edges(T, pos, edge_color='crimson', width=3, arrows=True, arrowsize=20, ax=ax)
labels = {v: f'{v}
({d[v]},{t[v]})' for v in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=11, font_weight='bold')
ax.set_title('Floresta DFS (tempos d,t nos rótulos)')
plt.show()

## 🔁 Exemplo 2: Digrafo

In [ ]:
D = nx.DiGraph()
D.add_edges_from([('A','B'),('A','C'),('B','D'),('C','D'),('D','E'),('B','F')])
cor, d, t, pai, arestas, ordem = dfs(D)
print('Ordem de descoberta:', ordem)
print('d:', d)
print('t:', t)
T = floresta_dfs_subgrafo(D, arestas)
pos = nx.spring_layout(D, seed=1)
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True, constrained_layout=True)
plt.subplot(1,2,1)
nx.draw(D, pos, with_labels=True, node_color='lightyellow', node_size=1000,
        font_size=12, font_weight='bold', arrows=True, arrowsize=20, ax=ax)
ax.set_title('Digrafo original')

plt.subplot(1,2,2)
nx.draw(D, pos, with_labels=True, node_color='lightgreen', node_size=1000,
        font_size=12, font_weight='bold', arrows=True, arrowsize=20, ax=ax)
nx.draw_networkx_edges(T, pos, edge_color='crimson', width=3, arrows=True, arrowsize=20, ax=ax)
labels = {v: f'{v}\n({d[v]},{t[v]})' for v in D.nodes()}
nx.draw_networkx_labels(D, pos, labels=labels, font_size=11, font_weight='bold')
ax.set_title('Floresta DFS (tempos d,t nos rótulos)')
plt.show()

## ⏱️ Complexidade de Tempo
- Inicialização e iteração pelos vértices: O(|V|).
- Cada aresta é considerada no máximo duas vezes em grafos não dirigidos (e uma vez em digrafos), totalizando O(|E|).
- Portanto, o tempo total é **O(|V| + |E|)**.

## 🧪 Experimento simples de custo (contadores)

In [ ]:
def dfs_com_contadores(G: nx.Graph):
    cor = {v: 'white' for v in G.nodes()}
    visitas_adj = 0
    tempo = 0
    def dfs_visit(u):
        nonlocal tempo, visitas_adj
        cor[u] = 'gray'
        tempo += 1
        for v in G.neighbors(u):
            visitas_adj += 1
            if cor[v] == 'white':
                dfs_visit(v)
        cor[u] = 'black'
        tempo += 1
    for u in G.nodes():
        if cor[u] == 'white':
            dfs_visit(u)
    return visitas_adj

G = nx.gnm_random_graph(100, 300, seed=7)
ops = dfs_com_contadores(G)
print('Visitas a adjacências (proporcional a |E|):', ops)